# Module 10 Assignment

In [2]:
# Step 1: Loading the Python libraries
import pymongo
import pandas as pd

In [4]:
# Step 2: Connecting to MongoDB
try:
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    if client:
        print("Connected to MongoDB server")
        client.server_info() # ping the server to verify the connection
except pymongo.errors.ConnectionFailure as e:
    print(f"Failed to connect to MongoDB server: {e}")

Connected to MongoDB server


In [6]:
# Step 3: Loading and Sampling the Yelp Dataset
business_data = pd.read_json('data/business_data_filtered.json', lines=True)
checkin_data = pd.read_json('data/checkin_data_filtered.json', lines=True)
review_data = pd.read_json('data/review_data_filtered.json', lines=True)
tip_data = pd.read_json('data/tip_data_filtered.json', lines=True)
user_data = pd.read_json('data/user_data_filtered.json', lines=True)

In [8]:
# Step 4: Inserting Data into MongoDB (0 points)
db = client['yelp']
business_collection = db['business_data']
checkin_collection = db['checkin_data']
review_collection = db['review_data']
tip_collection = db['tip_data']
user_collection = db['user_data']

# TODO
business_collection.insert_many(business_data.to_dict('records'))
checkin_collection.insert_many(checkin_data.to_dict('records'))
review_collection.insert_many(review_data.to_dict('records'))
tip_collection.insert_many(tip_data.to_dict('records'))
user_collection.insert_many(user_data.to_dict('records'))
print("Data inserted into MongoDB collections.")

Data inserted into MongoDB collections.


## Section 1: Querying Data from MongoDB

### Question #1 (10 points)

In [ ]:
# How many unique (distinct) users had the word "good" in at least one review they left?
# Make "good" case insensitive in the search ("good" = "Good" = "Good", etc.)
# Hint: There is a specific option collection option to filter out unique users.


#case-insensitive search for "good"
query_q1 = {"text": {"$regex": "good", "$options": "i"}}
#get unique user_ids
unique_users_q1 = review_collection.distinct("user_id", query_q1)
 #count the number of unique users
num_unique_users = len(unique_users_q1)
print(f"Number of unique users with 'good' in their reviews: {num_unique_users}")

### Question #2 (10 points)

In [ ]:
# Print the first 5 documents for one of these unique users from Question #1 which have reviews of 4 stars or more. There may be less than 5 documents - this is fine.

#get the first user from Q1 list
first_user_id_q1 = unique_users_q1[0] 
query_q2 = {'user_id': first_user_id_q1, 'stars': {'$gte': 4}}
docs_q2 = review_collection.find(query_q2).limit(5)
print(f"Reviews with 4+ stars for user {first_user_id_q1} (limit 5):")
for doc in docs_q2:
    print(doc)

### Question #3 (10 points)

In [ ]:
# Find a user ID in the tip dataframe with the word "great" in the "text" field.
# Find all businesses in the tip dataframe associated with that user ID.  
# Print the documents from the tips collection for those businesses.
# Set the limit of the print_query function to 10, although you may have less than 10 documents.


#case-insensitive search for "great"
query_q3_user = {"text": {"$regex": "great", "$options": "i"}}
#find one tip with "great"
tip_doc_q3 = tip_collection.find_one(query_q3_user)
if tip_doc_q3:
    user_id = tip_doc_q3["user_id"]  #get the user_id
    print(f"Found user {user_id} with 'great' in their tip.")
    #find all business_ids this user has tipped
    business_ids = tip_collection.distinct("business_id", {"user_id": user_id})
    #get all tips for those businesses limited to 10
    tips_query = {"business_id": {"$in": business_ids}}
    tips_q3 = tip_collection.find(tips_query).limit(10)
    print(f"Tips for businesses tipped by user {user_id}:")
    for tip in tips_q3:
        print(tip)
else:
    print("No tip with 'great' found")

## Section 2: ETL with MongoDB

### Question #4 (10 points)

In [ ]:
# Extract all reviews from the review_data collection using a MongoDB query as a Pandas dataframe.

In [ ]:
review_df = pd.DataFrame(list(review_collection.find()))
review_df 

### Question #5 (10 points)

In [ ]:
# Transform the data in the Pandas dataframe by selecting only the user_id, business_id, date, and stars fields and 
# rename the stars field to rating.

review_df = review_df[['user_id', 'business_id', 'date', 'stars']].rename(columns={'stars': 'rating'})
review_df

### Question #6 (10 points)

In [ ]:
# Convert the date field in the Pandas dataframe to a datetime object and extract the year and month into separate fields.

review_df["date"] = pd.to_datetime(review_df["date"])
review_df["year"] = review_df["date"].dt.year
review_df["month"] = review_df["date"].dt.month

### Question #7 (20 points)

In [ ]:
# Aggregate the data from review_df by year and month and calculate the average rating for each period.

review_agg = review_df.groupby(["year", "month"])["rating"].mean()
review_agg = review_agg.reset_index()

### Question #8 (10 points)

In [ ]:
# Write the aggregated dataframe to a new MongoDB collection called review_data_aggregated

review_data_aggregated = db["review_data_aggregated"]
review_data_aggregated.insert_many(review_agg.to_dict('records'))

### Question #9 (10 points)

In [ ]:
# Retrieve the first five documents in the collection

docs_q9 = review_data_aggregated.find({}).limit(5)
print("First 5 documents in review_data_aggregated:")
for doc in docs_q9:
    print(doc)

In [ ]:
# Close the MongoDB connection
client.close()